# Credit Risk and Loan Default Prediction
### Course: Introduction to Python Programming (College Project)
**Methodology:** Multiple Logistic Regression & Econometric Credit Underwriting  
**Dataset:** LendingClub Retail Loans (Accepted 2007–2018Q4)  
**Target:** Probability of Default (PD) for Retail Loan Applicants  

---

## Project Overview & Objectives
In retail banking and peer-to-peer lending, evaluating credit risk is the cornerstone of prudential risk management (Basel II/III Accord framework). When an applicant applies for an unsecured personal loan, the financial institution must accurately estimate their **Probability of Default (PD)**.

In this project, we develop an end-to-end Python pipeline using **Multiple Logistic Regression** to:
1. Ingest, sanitize, and preprocess raw LendingClub records while avoiding **Look-Ahead Bias**.
2. Engineer economic indicators such as the **Loan-to-Income (LTI)** ratio.
3. Partition data into stratified train/test folds (80/20) and standardize features without **Data Leakage**.
4. Train a logistic regression model with **balanced class weighting** to handle minority default prevalence.
5. Derive **Beta coefficients**, **Odds Ratios ($e^{\beta}$)**, and econometric standard errors ($p$-values).
6. Formulate a calibrated business pricing matrix that translates continuous PDs into **Credit Grades (A through F)** and **Offered Interest Rates (APRs)**.
7. Build an interactive applicant scoring function (`score_applicant`) for underwriting decisioning.

### Pipeline Flowchart
```
[ 1. Raw CSV Ingestion ] -> [ 2. Data Cleaning & Look-Ahead Filter ]
                                      │
                                      ▼
[ 4. 80/20 Stratified Split ] <- [ 3. Feature Engineering: LTI ]
             │
             ▼
[ 5. StandardScaler (Train Only) ] -> [ 6. Balanced Logistic Regression ]
                                                     │
                                                     ▼
[ 8. Grade (A-F) & APR Mapping ] <- [ 7. Probability Generation (PD) ]
             │
             ▼
[ 9. Applicant Underwriting & Scorecard Demo ]
```

## 1. Environment Setup & Configuration
We import essential libraries and define global business constants and credit grade cutoffs.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report
)

# Optional econometric library for p-values and standard errors
try:
    import statsmodels.api as sm
    HAS_STATSMODELS = True
except ImportError:
    HAS_STATSMODELS = False

# Visual formatting
sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline

DATA_FILE_PATH = "data/accepted_2007_2018Q4.csv"
OUTPUT_DIR = "outputs"
FINAL_FEATURES = ["annual_inc", "dti", "fico_range_low", "loan_amnt", "LTI"]

# Calibrated PD cutoffs for balanced logistic model
GRADE_CUTOFFS = {
    "A": 0.32,
    "B": 0.44,
    "C": 0.54,
    "D": 0.64,
    "E": 0.74,
    "F": 1.00
}

# Risk-based pricing schedule (APR mapping)
INTEREST_RATES = {
    "A": 0.0699,  # 6.99%
    "B": 0.0999,  # 9.99%
    "C": 0.1399,  # 13.99%
    "D": 0.1799,  # 17.99%
    "E": 0.2299,  # 22.99%
    "F": 0.2899   # 28.99%
}

print("Environment initialized successfully!")

## 2. Raw Data Ingestion with Exception Handling
We load only the 7 necessary columns via `usecols` to minimize memory usage, wrapped in a defensive `try-except` block.

In [ ]:
def load_data(file_path: str = DATA_FILE_PATH) -> pd.DataFrame:
    """Loads target CSV dataset with graceful exception handling and usecols."""
    if not os.path.exists(file_path):
        print(f"File '{file_path}' not found. Generating sample data...")
        try:
            import generate_sample_data
            generate_sample_data.create_sample_csv(file_path, n_rows=5000)
        except Exception as e:
            raise FileNotFoundError(f"Cannot find or generate dataset: {e}")
            
    try:
        header = pd.read_csv(file_path, nrows=2)
        candidates = ["id", "member_id", "loan_status", "annual_inc", "dti", "fico_range_low", "loan_amnt"]
        cols = [c for c in candidates if c in header.columns]
        df = pd.read_csv(file_path, usecols=cols, low_memory=False)
        print(f"Loaded {len(df):,} records with columns: {cols}")
        return df
    except FileNotFoundError as err:
        print(f"[ERROR] File not found: {err}")
        raise
    except pd.errors.ParserError as err:
        print(f"[ERROR] CSV parsing error: {err}")
        raise

df_raw = load_data(DATA_FILE_PATH)
df_raw.head()

## 3. Data Cleaning & Avoiding Look-Ahead Bias
### Why do we drop "Current" and "Late" loans?
**Look-Ahead Bias** occurs when information that would not have been known at the decision date is incorporated into model training. Loans currently in progress have not reached a terminal state; labeling a "Current" loan as safe is inaccurate because that borrower might default several months later. In banking, probability of default must be trained strictly on **resolved loan contracts**.
- `Fully Paid` $\rightarrow 0$ (Safe)
- `Charged Off` or `Default` $\rightarrow 1$ (Default)

In [ ]:
def clean_data(raw_df: pd.DataFrame) -> pd.DataFrame:
    """Cleans data, removes unresolved loans, imputes missing values, and clips outliers."""
    df = raw_df.copy()
    
    # 1. Standardize string column
    df["loan_status"] = df["loan_status"].astype(str).str.strip()
    
    # 2. Filter resolved contracts only
    resolved_mask = df["loan_status"].isin(["Fully Paid", "Charged Off", "Default"])
    df = df[resolved_mask].copy()
    
    # 3. Create binary target
    df["target"] = df["loan_status"].apply(lambda s: 0 if s == "Fully Paid" else 1)
    
    # 4. Drop identifier columns
    drop_cols = [c for c in ["id", "member_id", "loan_status"] if c in df.columns]
    df.drop(columns=drop_cols, inplace=True)
    
    # 5. Median imputation for numeric features
    numeric_cols = ["annual_inc", "dti", "fico_range_low", "loan_amnt"]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        if df[col].isna().sum() > 0:
            df[col] = df[col].fillna(df[col].median())
            
    # 6. Handle edge cases & division by zero
    df["annual_inc"] = df["annual_inc"].apply(lambda x: max(1000.0, float(x)))
    
    # 7. Clip extreme outliers
    inc_cap = float(df["annual_inc"].quantile(0.995))
    df["annual_inc"] = np.clip(df["annual_inc"], a_min=1000.0, a_max=inc_cap)
    
    dti_cap = min(45.0, float(df["dti"].quantile(0.99)))
    df["dti"] = np.clip(df["dti"], a_min=0.0, a_max=dti_cap)
    
    return df

df_clean = clean_data(df_raw)
print(f"Cleaned dataset has {len(df_clean):,} resolved records.")
df_clean.describe()

## 4. Feature Engineering: Loan-to-Income (LTI)
$$\text{LTI} = \frac{\text{loan\_amnt}}{\text{annual\_inc}}$$
LTI standardizes the borrowing magnitude relative to the borrower's earnings capacity.

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Calculates LTI ratio and bounds extreme values."""
    df["LTI"] = df["loan_amnt"] / df["annual_inc"]
    df["LTI"] = np.clip(df["LTI"], a_min=0.0, a_max=1.5)
    return df

df_feat = engineer_features(df_clean)
print("Engineered features:", FINAL_FEATURES)
df_feat[FINAL_FEATURES].head()

## 5. Data Partitioning & Normalization (StandardScaler)
### Avoiding Data Leakage:
The `StandardScaler` must be fit **strictly on the training partition (`X_train`)**, and then used to transform both `X_train` and `X_test`. If fit on the test set, information from the evaluation fold would leak into the training process.

In [ ]:
X = df_feat[FINAL_FEATURES]
y = df_feat["target"]

# 80/20 Stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Full dataset default rate: {(y.mean()*100):.2f}%")
print(f"Train default rate:        {(y_train.mean()*100):.2f}%")
print(f"Test default rate:         {(y_test.mean()*100):.2f}%")

# Normalization: Fit on Train ONLY
scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=FINAL_FEATURES, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=FINAL_FEATURES, index=X_test.index)

X_train_scaled.head()

## 6. Multiple Logistic Regression Model Training
We use `class_weight="balanced"` to penalize misclassifying default contracts, which form a ~15-20% minority.

In [ ]:
clf = LogisticRegression(class_weight="balanced", random_state=42, max_iter=1000)
clf.fit(X_train_scaled, y_train)
print("Model training complete!")

# Econometric Statsmodels Logit (Bonus)
if HAS_STATSMODELS:
    X_train_sm = sm.add_constant(X_train_scaled)
    sm_model = sm.Logit(y_train, X_train_sm).fit(disp=False)
    display(sm_model.summary())
else:
    sm_model = None

## 7. Econometric Interpretation: Beta Coefficients & Odds Ratios
$$\text{Odds Ratio} = e^{\beta_i}$$
- If $\text{Odds Ratio} > 1$: Feature increases the odds of default.
- If $\text{Odds Ratio} < 1$: Feature protects against default.

In [ ]:
coef_table = pd.DataFrame({
    "Feature": FINAL_FEATURES,
    "Beta (Std)": clf.coef_[0],
    "Odds Ratio (e^Beta)": np.exp(clf.coef_[0])
})

coef_table["Interpretation"] = coef_table.apply(
    lambda row: f"Each 1-SD increase in {row['Feature']} modifies default odds by {(row['Odds Ratio (e^Beta)'] - 1)*100:+.1f}%.",
    axis=1
)

display(coef_table)

## 8. Model Evaluation & ROC-AUC
We evaluate discrimination performance on the unseen test set.

In [ ]:
y_prob = clf.predict_proba(X_test_scaled)[:, 1]
y_pred = clf.predict(X_test_scaled)

auc = roc_auc_score(y_test, y_prob)
print(f"Test Set ROC-AUC Score: {auc:.4f} ({auc*100:.2f}%)")

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix (at 0.50 cutoff):\n", cm)
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=["Safe (0)", "Default (1)"]))

## 9. Business Logic Mapping: Grades (A-F) & Pricing APR
We convert continuous PDs into letter grades and verify risk monotonicity.

In [ ]:
def assign_grade(pd_val: float) -> str:
    if pd_val < GRADE_CUTOFFS["A"]:
        return "A"
    elif pd_val < GRADE_CUTOFFS["B"]:
        return "B"
    elif pd_val < GRADE_CUTOFFS["C"]:
        return "C"
    elif pd_val < GRADE_CUTOFFS["D"]:
        return "D"
    elif pd_val < GRADE_CUTOFFS["E"]:
        return "E"
    else:
        return "F"

eval_df = pd.DataFrame({"actual": y_test.values, "pd": y_prob})
eval_df["grade"] = eval_df["pd"].apply(assign_grade)

grade_summary = eval_df.groupby("grade").agg(
    applicants=("actual", "count"),
    avg_pd=("pd", "mean"),
    actual_default_rate=("actual", "mean")
).reindex(["A", "B", "C", "D", "E", "F"]).fillna(0)

grade_summary["offered_apr"] = [INTEREST_RATES[g] for g in grade_summary.index]
display(grade_summary)

## 10. Single Applicant Underwriting Function (`score_applicant`)

In [ ]:
def score_applicant(profile: dict) -> dict:
    """Scores an individual applicant profile and returns underwriting decision."""
    inc = float(max(1000.0, profile["annual_inc"]))
    loan = float(profile["loan_amnt"])
    dti = float(profile["dti"])
    fico = float(profile["fico_range_low"])
    lti = loan / inc
    
    row = pd.DataFrame([{
        "annual_inc": inc,
        "dti": dti,
        "fico_range_low": fico,
        "loan_amnt": loan,
        "LTI": lti
    }])[FINAL_FEATURES]
    
    scaled_row = scaler.transform(row)
    pd_val = float(clf.predict_proba(scaled_row)[0, 1])
    grade = assign_grade(pd_val)
    apr = INTEREST_RATES[grade]
    
    return {
        "PD": pd_val,
        "Grade": grade,
        "APR": apr,
        "Decision": "Approved" if grade in ["A", "B", "C", "D"] else "Declined / High Risk"
    }

sample_profiles = [
    {"name": "Low Risk Prime", "annual_inc": 120000, "dti": 10.5, "fico_range_low": 780, "loan_amnt": 10000},
    {"name": "Medium Risk", "annual_inc": 55000, "dti": 20.0, "fico_range_low": 680, "loan_amnt": 15000},
    {"name": "High Risk Subprime", "annual_inc": 28000, "dti": 36.0, "fico_range_low": 610, "loan_amnt": 25000}
]

for p in sample_profiles:
    res = score_applicant(p)
    print(f"{p['name']} -> PD: {res['PD']:.2%}, Grade: {res['Grade']}, APR: {res['APR']*100:.2f}%, Decision: {res['Decision']}")

## 11. Visualizations
We plot the ROC curve, Sigmoid curve, Feature coefficients, and Grade calibration.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[0].plot(fpr, tpr, color="#2f9e44", lw=2.5, label=f"Logistic Regression (AUC = {auc:.3f})")
axes[0].plot([0, 1], [0, 1], color="gray", linestyle="--")
axes[0].set_title("ROC Curve (Discrimination Ability)", fontweight="bold")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend(loc="lower right")

# 2. Coefficients Bar Chart
colors = ["#2b8a3e" if c < 0 else "#c92a2a" for c in clf.coef_[0]]
axes[1].barh(FINAL_FEATURES, clf.coef_[0], color=colors)
axes[1].axvline(0, color="black", lw=1)
axes[1].set_title("Standardized Logistic Regression Beta Coefficients", fontweight="bold")
axes[1].set_xlabel("Beta Impact on Log-Odds")

plt.tight_layout()
plt.show()